In [1]:
# evaluate_generalization.py
import os
import glob
import cv2
import numpy as np
import xml.etree.ElementTree as ET
import torch
import torch.nn as nn
from stain_utils import MacenkoNormalizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Architecture Definition (Same U-Net as trained on CoNSeP) ---
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.inc = DoubleConv(3, 32)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(32, 64))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.up1 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv_up1 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv_up2 = DoubleConv(256, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv_up3 = DoubleConv(128, 64)
        self.up4 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv_up4 = DoubleConv(64, 32)
        self.outc = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.conv_up1(torch.cat([self.up1(x5), x4], dim=1))
        x = self.conv_up2(torch.cat([self.up2(x), x3], dim=1))
        x = self.conv_up3(torch.cat([self.up3(x), x2], dim=1))
        x = self.conv_up4(torch.cat([self.up4(x), x1], dim=1))
        return self.outc(x)

def parse_xml_to_mask(xml_path, shape):
    """Converts MoNuSeg polygon vertices into ground truth binary mask."""
    mask = np.zeros(shape[:2], dtype=np.uint8)
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for region in root.iter("Region"):
        coords = []
        for vertex in region.iter("Vertex"):
            x = float(vertex.attrib["X"])
            y = float(vertex.attrib["Y"])
            coords.append([int(round(x)), int(round(y))])
        if len(coords) > 2:
            pts = np.array(coords, dtype=np.int32)
            cv2.fillPoly(mask, [pts], 1)
    return mask

def compute_metrics(pred_mask, gt_mask):
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    iou = (intersection + 1e-7) / (union + 1e-7)
    dice = (2.0 * intersection + 1e-7) / (pred_mask.sum() + gt_mask.sum() + 1e-7)
    return dice, iou

def predict_large_image(model, img_rgb, patch_size=256, stride=128):
    """Sliding-window tiling with Gaussian averaging for clean seamless output."""
    h, w, _ = img_rgb.shape
    prob_map = np.zeros((h, w), dtype=np.float32)
    count_map = np.zeros((h, w), dtype=np.float32)

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            patch = img_rgb[y:y+patch_size, x:x+patch_size].astype(np.float32) / 255.0
            t = torch.tensor(patch.transpose(2, 0, 1)).unsqueeze(0).to(device)
            with torch.no_grad():
                pred = torch.sigmoid(model(t)).squeeze().cpu().numpy()
            prob_map[y:y+patch_size, x:x+patch_size] += pred
            count_map[y:y+patch_size, x:x+patch_size] += 1.0

    count_map[count_map == 0] = 1.0
    final_prob = prob_map / count_map
    return (final_prob > 0.5).astype(np.uint8)

def run_benchmark():
    # 1. Load CoNSeP-trained model
    model = UNet().to(device)
    model.load_state_dict(torch.load("models/unet.pth", map_location=device, weights_only=True))
    model.eval()

    normalizer = MacenkoNormalizer()
    img_dir = "dataset/MoNuSeg_test/images"
    xml_dir = "dataset/MoNuSeg_test/annotations"
    img_paths = sorted(glob.glob(os.path.join(img_dir, "*.*")))

    if not img_paths:
        print("No images found in dataset/MoNuSeg_test/images! Place your MoNuSeg test files there.")
        return

    raw_dices, raw_ious = [], []
    norm_dices, norm_ious = [], []

    print(f"Evaluating CoNSeP-trained U-Net on {len(img_paths)} unseen MoNuSeg slides...\n")

    for p in img_paths:
        base = os.path.splitext(os.path.basename(p))[0]
        xml_p = os.path.join(xml_dir, f"{base}.xml")
        if not os.path.exists(xml_p):
            continue

        orig_bgr = cv2.imread(p)
        orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
        gt_mask = parse_xml_to_mask(xml_p, orig_rgb.shape)

        # A: Zero-Shot without normalization
        pred_raw = predict_large_image(model, orig_rgb)
        d_raw, i_raw = compute_metrics(pred_raw, gt_mask)
        raw_dices.append(d_raw)
        raw_ious.append(i_raw)

        # B: Zero-Shot WITH Macenko Stain Normalization
        norm_rgb = normalizer.normalize(orig_rgb)
        pred_norm = predict_large_image(model, norm_rgb)
        d_norm, i_norm = compute_metrics(pred_norm, gt_mask)
        norm_dices.append(d_norm)
        norm_ious.append(i_norm)

        print(f"Slide: {base[:20]}... | Raw Dice: {d_raw:.3f} -> Normalized Dice: {d_norm:.3f}")

    print("\n" + "="*60)
    print("📊 CROSS-DATASET GENERALIZATION EXPERIMENTAL RESULTS")
    print("="*60)
    print(f"Source Training Domain : CoNSeP (Colorectal Tissue)")
    print(f"Target Testing Domain  : MoNuSeg (Multi-Organ Tissue: Breast, Kidney, Liver, etc.)")
    print("-" * 60)
    print(f"Mean Dice (Direct Domain Transfer / No Norm) : {np.mean(raw_dices):.4f}")
    print(f"Mean IoU  (Direct Domain Transfer / No Norm) : {np.mean(raw_ious):.4f}")
    print(f"Mean Dice (With Macenko Stain Normalization) : {np.mean(norm_dices):.4f}")
    print(f"Mean IoU  (With Macenko Stain Normalization) : {np.mean(norm_ious):.4f}")
    print(f"Performance Gain from Normalization         : +{(np.mean(norm_dices) - np.mean(raw_dices))*100:.2f}% Dice")
    print("="*60)

if __name__ == "__main__":
    run_benchmark()

ModuleNotFoundError: No module named 'stain_utils'